# PyMC-BART Wrapper: Multiclass Categorical and Ordinal Classification

This notebook demonstrates the `BARTModelWrapper` class from `pymc_bart_wrapper.py` for two ways of building classification models on the `modelDataDeIdentified.csv` dataset by target variable type:

1. **Categorical (unordered multiclass):** Uses `pmb.BART` → softmax → `pm.Categorical` likelihood
2. **Ordinal (ordered classes):** Uses `pmb.BART` (1-D latent) → `pm.OrderedLogistic` with learned cutpoints

We walk through the full workflow: data loading, preprocessing (one-hot encoding, missing value handling), data registration, model fitting, out-of-sample prediction, and result visualization. Along the way, we explain key concepts including:

- How categorical variables are encoded for PyMC-BART
- The purpose of `register_data()` for pre-fitting encoders
- How `X_shared_` (a `pm.Data` container) enables out-of-sample prediction
- What "draw from the full generative model (BART trees + likelihood jointly)" means

> **Reference:** The wrapper follows the model specification from the [PyMC-BART categorical regression example](https://www.pymc.io/projects/bart/en/latest/examples/bart_categorical_hawks.html).

## 1. Import Libraries and Load Data

We import the necessary libraries: pandas and numpy for data handling, matplotlib and seaborn for visualization, scikit-learn for splitting and evaluation metrics, arviz for Bayesian diagnostics, and our custom `BARTModelWrapper` class.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import arviz as az
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from pymc_bart_wrapper import BARTModelWrapper

%matplotlib inline
sns.set_style("whitegrid")

In [ ]:
# Load the dataset (supports both CSV and Parquet)
DATA_PATH = Path('./data/modelDataDeIdentified.parquet')
if DATA_PATH.suffix == '.csv':
    df = pd.read_csv(DATA_PATH)
elif DATA_PATH.suffix == '.parquet':
    df = pd.read_parquet(DATA_PATH)
else:
    raise ValueError(f"Unsupported file format: {DATA_PATH.suffix}")

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

## 2. Explore the Dataset

Let's examine the dataset: view the first few rows, check the unique target classes in `finalized_label`, and visualize the class distribution.

In [ ]:
df.head()

In [ ]:
print(f"Target classes: {df['finalized_label'].unique().tolist()}")
print(f"\nTarget distribution:")
print(df['finalized_label'].value_counts())

In [ ]:
# Bar chart of target class distribution
fig, ax = plt.subplots(figsize=(10, 5))
target_counts = df['finalized_label'].value_counts()
bars = ax.bar(target_counts.index, target_counts.values, color="steelblue", edgecolor="black")
ax.set_xlabel("Target Class")
ax.set_ylabel("Count")
ax.set_title("Target Class Distribution (finalized_label)")
plt.xticks(rotation=45, ha="right")
for bar, count in zip(bars, target_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            str(count), ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

## 3. Define Variables and Train/Test Split

We define the **target variable** (`finalized_label`), the **predictor variables** (a mix of numeric and categorical features), and the **non-numeric variables** that require one-hot encoding. Then we perform a stratified 80/20 train/test split to preserve class proportions.

In [ ]:
TARGET = "finalized_label"

PREDICTOR_VARS = [
    # Numeric predictors
    "pat_age_at_test",
    "hgb_a", "hgb_f", "hgb_s", "hgb_c",
    "hgb_a2", "hgb_a2_variant",
    "hgb_e", "hgb_barts", "hgb_h", "hgb_d",
    "hgb_other_hgb", "total_hgb_count",
    # CBC test results
    "rbc_mean", "hgb_mean", "hct_mean", "mcv_mean", "rdw_cv_mean",
    # Categorical predictors
    "sex",
    "hgb_a_category", "hgb_s_category", "hgb_c_category",
]

NON_NUMERIC_VARS = [
    "sex",
    "hgb_a_category", "hgb_s_category", "hgb_c_category",
]

print(f"Target variable: {TARGET}")
print(f"Number of predictors: {len(PREDICTOR_VARS)}")
print(f"  Numeric: {len(PREDICTOR_VARS) - len(NON_NUMERIC_VARS)}")
print(f"  Categorical (to be one-hot encoded): {len(NON_NUMERIC_VARS)}")
print(f"  Non-numeric vars: {NON_NUMERIC_VARS}")

In [ ]:
# Stratified train/test split (80/20)
df_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df[TARGET]
)

print(f"Train size: {len(df_train)}")
print(f"Test  size: {len(df_test)}")

## 4. Understanding Data Preprocessing: Encoding Categorical Variables

**PyMC-BART does not natively handle categorical (string) features.** The `BARTModelWrapper` addresses this by applying **one-hot encoding** via scikit-learn's `OneHotEncoder` to all columns listed in `non_numeric_vars`.

The preprocessing pipeline in `preprocess()` works as follows:

1. **Numeric columns** with missing values are filled with a sentinel value (default `-99`) when `fill_missing=True`. This places missing values outside the normal data range so the BART trees can learn to split on "missingness" as a feature.
2. **Categorical columns** with missing values are filled with a new `"missing"` category, which the one-hot encoder then treats as just another level.
3. If `fill_missing=False`, rows containing **any** NaN (across target + selected predictors) are dropped instead.
4. Categorical columns are **one-hot encoded** — each unique category becomes a binary (0/1) column. For example, `sex` with values `["M", "F"]` becomes two columns: `sex_F` and `sex_M`.
5. The **target variable** is integer-encoded (factorized) so each class label maps to a code `0, 1, 2, ...`.

Let's demonstrate this by instantiating a wrapper and calling `preprocess()` directly to see the transformed data.

In [ ]:
# Demonstrate preprocessing: instantiate a temporary wrapper and call preprocess()
demo_wrapper = BARTModelWrapper(
    target_var=TARGET,
    predictor_vars=PREDICTOR_VARS,
    non_numeric_vars=NON_NUMERIC_VARS,
    target_type="categorical",
    fill_missing=True,
    missing_numeric_fill=-99,
)

# preprocess with fit=True learns encoders and returns (X, y_codes)
X_demo, y_codes_demo = demo_wrapper.preprocess(df_train, fit=True)

print(f"Processed predictor matrix X shape: {X_demo.shape}")
print(f"Original predictor count: {len(PREDICTOR_VARS)}")
print(f"After one-hot encoding:   {X_demo.shape[1]} columns")
print(f"\nColumns after encoding:\n{X_demo.columns.tolist()}")
print(f"\nUnique target codes: {np.unique(y_codes_demo)}")
print(f"Category mapping: {demo_wrapper.category_map_}")

In [ ]:
# Show the first few rows of the processed predictor matrix
X_demo.head()

## 5. Understanding Data Registration with `register_data()`

The `register_data(df)` method **pre-fits the one-hot encoder and target encoding on the full dataset** before the train/test split. This is an important step because:

- **Every category is represented:** If a rare category appears only in the test set, the one-hot encoder (fitted only on the training set) would not know about it and would produce an all-zeros row. Pre-fitting on the full data avoids this.
- **Consistent encoding:** The target variable encoding (mapping class labels → integer codes) is the same whether we use it for training or prediction.
- **Optional but recommended:** If `register_data()` is never called, encoders are learned from the training set only during `fit()`.

Internally, `register_data()`:
1. Identifies numeric vs. categorical predictor columns.
2. Fills missing categorical values with `"missing"` (if `fill_missing=True`).
3. Learns the target encoding (`category_codes_`, `category_map_`, `n_classes_`).
4. Fits the `OneHotEncoder` on all categorical predictor columns from the full dataset.
5. Sets `encoder_fitted_ = True` so that `preprocess()` and `fit()` reuse these pre-fitted encoders.

In [ ]:
# Demonstrate register_data(): pre-fit encoders on the FULL dataset
reg_demo = BARTModelWrapper(
    target_var=TARGET,
    predictor_vars=PREDICTOR_VARS,
    non_numeric_vars=NON_NUMERIC_VARS,
    target_type="categorical",
    fill_missing=True,
)

# Register the full dataset
reg_demo.register_data(df)

# Inspect what was learned
print(f"encoder_fitted_: {reg_demo.encoder_fitted_}")
print(f"\nTarget classes (category_codes_):\n  {reg_demo.category_codes_.tolist()}")
print(f"\nCategory map (code → label):\n  {reg_demo.category_map_}")
print(f"\nn_classes_: {reg_demo.n_classes_}")
print(f"\nOne-hot encoded columns (ohe_columns_):\n  {reg_demo.ohe_columns_}")

---

## 6. Example A: Multiclass Categorical BART Model – Setup and Fit

For `target_type="categorical"`, the wrapper builds a PyMC model with the following structure:

1. **`pmb.BART("mu", X, y, m=50, dims=["classes", "n_obs"])`** — The BART prior produces a matrix of latent scores `mu` with shape `(n_classes, n_obs)`. Each element represents the "log-odds-like" score for a given class and observation.
2. **`theta = softmax(mu, axis=0)`** — The softmax function converts the raw latent scores into proper class probabilities that sum to 1 across classes for each observation.
3. **`pm.Categorical("y", p=theta.T, observed=y_codes)`** — The categorical likelihood draws observed class labels from the class probability vector.

The number of trees `m=50` controls model complexity. We use `chains=2, draws=500, tune=500` for faster demo execution (increase for production use).

In [ ]:
# Instantiate the categorical wrapper
cat_wrapper = BARTModelWrapper(
    target_var=TARGET,
    predictor_vars=PREDICTOR_VARS,
    non_numeric_vars=NON_NUMERIC_VARS,
    target_type="categorical",       # unordered multiclass
    fill_missing=True,               # fill NaN with missing_numeric_fill / "missing"
    missing_numeric_fill=-99,        # custom fill value for missing numerics
)

# Pre-fit encoders on the full dataset so every category is known
cat_wrapper.register_data(df)
print(cat_wrapper)

In [ ]:
# Fit the categorical BART model
# (reduced draws/trees for demo speed; increase for real analysis)
cat_wrapper.fit(
    df_train,
    m=50,
    chains=2,
    draws=500,
    tune=500,
    random_seed=42,
)
print("Categorical model fitted.")

## 7. Example A: Inspect Inference Data and Posterior Summary

After fitting, we can retrieve the ArviZ `InferenceData` object, which contains the posterior samples, posterior predictive draws, and other diagnostics. The `summary()` method provides a convenient table of posterior statistics.

In [ ]:
# Retrieve the ArviZ InferenceData object
idata_cat = cat_wrapper.get_inference_data()
print(idata_cat)

In [ ]:
# Posterior summary table
cat_wrapper.summary()

## 8. Example A: Out-of-Sample Prediction

We now generate predictions on the held-out test set using `cat_wrapper.predict()`. This method:

1. Preprocesses `df_test` using the same encoding learned during registration/training.
2. Swaps the shared covariate matrix with the test data.
3. Draws from the full posterior predictive distribution.
4. Returns predicted labels, class probabilities, and raw posterior predictive draws.

In [ ]:
# Predict on held-out test data
results_cat = cat_wrapper.predict(df_test, random_seed=42)

print(f"Predicted labels (first 10): {results_cat['predicted_labels'][:10]}")
print(f"Class prob mean shape: {results_cat['class_prob_mean'].shape}")
print(f"Posterior predictive shape: {results_cat['posterior_predictive'].shape}")

In [ ]:
# Display predicted class probabilities for the first 5 test observations
prob_df = pd.DataFrame(
    results_cat['class_prob_mean'][:5],
    columns=cat_wrapper.category_codes_,
    index=[f"Obs {i}" for i in range(5)]
)
print("Posterior mean class probabilities (first 5 test observations):")
prob_df

## 9. Understanding `X_shared_` and `pm.sample_posterior_predictive`

### What is `X_shared_`?

`X_shared_` is a **`pm.Data` container** — a PyTensor shared variable created during model construction that holds the covariate (predictor) matrix. In the model-building code, it is created as:

```python
X_shared = pm.Data("X", X.values)
```

Because it is a **shared variable**, its values can be **swapped in-place** without rebuilding the model graph. The entire model graph — BART trees, softmax/cutpoints, and the likelihood — references this shared variable. This is the key mechanism that enables out-of-sample prediction.

### What does "Swap the shared covariate matrix with new data" mean?

In the `predict()` method, the line:

```python
self.X_shared_.set_value(X_new.values)
```

**replaces the training covariate matrix** (stored in the shared variable during `fit()`) **with the test/new data matrix**. Since the model graph holds a reference to this shared variable (not a copy of the data), all downstream computations — BART tree predictions, softmax/cutpoint transforms, and the likelihood — now operate on the new data without any changes to the model structure.

### What does "draw from the full generative model (BART trees + likelihood jointly)" mean?

After swapping in the new data, `pm.sample_posterior_predictive` traverses the **full generative model** for each posterior sample:

1. **BART trees:** Take one posterior draw of the BART tree ensemble parameters (tree structures, leaf values). Evaluate these trees on the **new** test X to produce latent scores `mu` for each test observation.

2. **Link function:** Apply the appropriate transformation:
   - **Categorical model:** `softmax(mu)` converts the latent scores matrix into class probabilities `theta`.
   - **Ordinal model:** The cumulative logistic CDF with learned cutpoints converts the 1-D latent score into ordinal class probabilities.

3. **Likelihood:** Draw an observation `y` from the likelihood distribution:
   - **Categorical:** `y ~ Categorical(p=theta)` — sample a class label from the class probabilities.
   - **Ordinal:** `y ~ OrderedLogistic(eta=mu, cutpoints=c)` — sample an ordinal class.

**"BART + likelihood jointly"** means that each posterior predictive draw uses **one consistent posterior sample of all model parameters** (BART trees *and* cutpoints/other parameters). This propagates the **full posterior uncertainty** — uncertainty in the tree structure, leaf values, and any additional parameters — through the entire generative pipeline, rather than just point-predicting `mu` and plugging it in. The result is a distribution of predicted class labels for each test observation, from which we compute empirical class probabilities and point predictions.

In [ ]:
# Inspect the X_shared_ attribute
print(f"Type of X_shared_: {type(cat_wrapper.X_shared_)}")
print(f"Current X_shared_ shape (after predict swapped in test data): "
      f"{cat_wrapper.X_shared_.get_value().shape}")
print(f"\nThis shared variable is referenced by the model graph.")
print(f"When set_value() replaces its contents, all downstream")
print(f"computations (BART → softmax → Categorical) automatically")
print(f"operate on the new data.")

## 10. Example A: Visualize Categorical Model Results

Let's evaluate the categorical model's predictions with a confusion matrix, a classification report, and visualizations of the predicted class probabilities.

In [ ]:
# Confusion matrix for categorical model
y_true_cat = df_test[TARGET].values
y_pred_cat = results_cat["predicted_labels"]

cm_cat = confusion_matrix(y_true_cat, y_pred_cat, labels=cat_wrapper.category_codes_)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_cat, annot=True, fmt="d", cmap="Blues",
            xticklabels=cat_wrapper.category_codes_,
            yticklabels=cat_wrapper.category_codes_, ax=ax)
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
ax.set_title("Confusion Matrix – Categorical BART Model")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print("\nClassification Report – Categorical Model:")
print(classification_report(y_true_cat, y_pred_cat, target_names=list(cat_wrapper.category_codes_)))

In [ ]:
# Posterior predictive class probabilities for a single test observation
obs_idx = 0
probs = results_cat["class_prob_mean"][obs_idx]
classes = list(cat_wrapper.category_codes_)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(classes, probs, color="steelblue", edgecolor="black")
ax.set_xlabel("Class")
ax.set_ylabel("Posterior Mean Probability")
ax.set_title(f"Predicted Class Probabilities for Test Observation {obs_idx}\n"
             f"(True label: {df_test[TARGET].iloc[obs_idx]})")
plt.xticks(rotation=45, ha="right")
for bar, p in zip(bars, probs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{p:.3f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Compare true vs predicted label distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
classes = list(cat_wrapper.category_codes_)

true_counts = pd.Series(y_true_cat).value_counts().reindex(classes, fill_value=0)
pred_counts = pd.Series(y_pred_cat).value_counts().reindex(classes, fill_value=0)

axes[0].bar(classes, true_counts, color="steelblue", edgecolor="black")
axes[0].set_title("True Label Distribution (Test Set)")
axes[0].set_xlabel("Class")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(classes, pred_counts, color="coral", edgecolor="black")
axes[1].set_title("Predicted Label Distribution (Categorical Model)")
axes[1].set_xlabel("Class")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

---

## 11. Example B: Ordinal BART Model – Setup and Fit

For `target_type="ordinal"`, the wrapper builds a fundamentally different model that respects the **ordering** of the outcome classes:

1. **`pmb.BART("mu", X, y, m=50)`** — BART produces a **single 1-D latent score** `mu` per observation (not a matrix). Higher `mu` means the observation is more likely to belong to a higher-order class.

2. **`pm.Normal("cutpoints", ..., transform=ordered)`** — A set of `n_classes - 1` learned cutpoints partition the real line into `n_classes` bins. The `ordered` transform ensures `cutpoints[0] < cutpoints[1] < ...`.

3. **`pm.OrderedLogistic("y", eta=mu, cutpoints=cutpoints)`** — The ordinal likelihood uses the cumulative logistic CDF to compute the probability of each ordinal class: $P(Y \le k) = \text{logistic}(c_k - \mu)$, and class probabilities are obtained by differencing adjacent cumulative probabilities.

### Defining the ordinal order

For ordinal modelling, we need a **meaningful low-to-high ordering** of the diagnosis labels. Here we define one based on clinical severity:

In [ ]:
# Define the ordinal ordering (low → high clinical severity)
ORDINAL_ORDER = [
    "Normal",
    "Hemoglobin_C_Trait",
    "Sickle_Cell_Trait",
    "Beta_Thalassemia",
    "HGB_SC_Disease",
    "Sickle_Cell_Disease",
    "Other",
]

# Instantiate the ordinal wrapper
ord_wrapper = BARTModelWrapper(
    target_var=TARGET,
    predictor_vars=PREDICTOR_VARS,
    non_numeric_vars=NON_NUMERIC_VARS,
    target_type="ordinal",           # ordered outcome
    ordinal_order=ORDINAL_ORDER,     # explicit severity ordering
    fill_missing=True,
)

# Pre-fit encoders on the full dataset
ord_wrapper.register_data(df)
print(ord_wrapper)

In [ ]:
# Fit the ordinal BART model
ord_wrapper.fit(
    df_train,
    m=50,
    chains=2,
    draws=500,
    tune=500,
    random_seed=42,
)
print("Ordinal model fitted.")

## 12. Example B: Out-of-Sample Prediction

Generate predictions from the ordinal model on the same test set.

In [ ]:
# Predict on held-out test data with the ordinal model
results_ord = ord_wrapper.predict(df_test, random_seed=42)

print(f"Ordinal – predicted labels (first 10): {results_ord['predicted_labels'][:10]}")
print(f"Ordinal – class prob mean shape: {results_ord['class_prob_mean'].shape}")

In [ ]:
# Display predicted class probabilities for the first 5 test observations (ordinal)
prob_df_ord = pd.DataFrame(
    results_ord['class_prob_mean'][:5],
    columns=ORDINAL_ORDER,
    index=[f"Obs {i}" for i in range(5)]
)
print("Posterior mean class probabilities – Ordinal Model (first 5 test observations):")
prob_df_ord

## 13. Example B: Visualize Ordinal Model Results

We evaluate the ordinal model with the same visualizations, using the ordinal class ordering. We also plot cumulative predicted probabilities for a single observation, which is a natural visualization for ordinal models.

In [ ]:
# Confusion matrix for ordinal model
y_true_ord = df_test[TARGET].values
y_pred_ord = results_ord["predicted_labels"]

cm_ord = confusion_matrix(y_true_ord, y_pred_ord, labels=ORDINAL_ORDER)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_ord, annot=True, fmt="d", cmap="Greens",
            xticklabels=ORDINAL_ORDER,
            yticklabels=ORDINAL_ORDER, ax=ax)
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
ax.set_title("Confusion Matrix – Ordinal BART Model")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print("\nClassification Report – Ordinal Model:")
print(classification_report(y_true_ord, y_pred_ord, target_names=ORDINAL_ORDER))

In [ ]:
# Cumulative predicted probabilities for the ordinal model (single observation)
obs_idx = 0
probs_ord = results_ord["class_prob_mean"][obs_idx]
cum_probs = np.cumsum(probs_ord)

fig, ax = plt.subplots(figsize=(10, 5))
ax.step(range(len(ORDINAL_ORDER)), cum_probs, where="mid",
        color="darkgreen", linewidth=2, marker="o")
ax.set_xticks(range(len(ORDINAL_ORDER)))
ax.set_xticklabels(ORDINAL_ORDER, rotation=45, ha="right")
ax.set_xlabel("Ordinal Class (low → high severity)")
ax.set_ylabel("Cumulative Probability")
ax.set_title(f"Cumulative Class Probabilities – Ordinal Model\n"
             f"Test Observation {obs_idx} (True: {df_test[TARGET].iloc[obs_idx]})")
ax.axhline(0.5, color="gray", linestyle="--", alpha=0.5, label="P = 0.5")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Per-class accuracy for the ordinal model
per_class_acc_ord = {}
for cls in ORDINAL_ORDER:
    mask = y_true_ord == cls
    if mask.sum() > 0:
        per_class_acc_ord[cls] = (np.array(y_pred_ord)[mask] == y_true_ord[mask]).mean()
    else:
        per_class_acc_ord[cls] = 0.0

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(per_class_acc_ord.keys(), per_class_acc_ord.values(),
              color="darkgreen", edgecolor="black")
ax.set_xlabel("Class")
ax.set_ylabel("Accuracy")
ax.set_title("Per-Class Accuracy – Ordinal BART Model")
ax.set_ylim(0, 1.05)
plt.xticks(rotation=45, ha="right")
for bar, acc in zip(bars, per_class_acc_ord.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{acc:.2f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

---

## 14. Accuracy Comparison Between Categorical and Ordinal Models

Let's compute and compare the overall accuracy and per-class accuracy of both models on the same test set.

In [ ]:
# Overall accuracy comparison
y_true_labels = df_test[TARGET].values
acc_cat = accuracy_score(y_true_labels, results_cat["predicted_labels"])
acc_ord = accuracy_score(y_true_labels, results_ord["predicted_labels"])

print(f"Categorical accuracy: {acc_cat:.3f}")
print(f"Ordinal     accuracy: {acc_ord:.3f}")

# Per-class accuracy for both models
all_classes = list(cat_wrapper.category_codes_)
per_class_cat = {}
per_class_ord = {}
for cls in all_classes:
    mask = y_true_labels == cls
    if mask.sum() > 0:
        per_class_cat[cls] = (np.array(results_cat["predicted_labels"])[mask] == y_true_labels[mask]).mean()
        per_class_ord[cls] = (np.array(results_ord["predicted_labels"])[mask] == y_true_labels[mask]).mean()
    else:
        per_class_cat[cls] = 0.0
        per_class_ord[cls] = 0.0

comparison_df = pd.DataFrame({
    "Categorical Accuracy": per_class_cat,
    "Ordinal Accuracy": per_class_ord,
})
comparison_df.loc["Overall"] = [acc_cat, acc_ord]
comparison_df

## 15. Visualize Accuracy Comparison

A side-by-side comparison of per-class accuracy for both models, plus an overall accuracy summary.

In [ ]:
# Grouped bar chart: per-class accuracy comparison
x = np.arange(len(all_classes))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - width/2, [per_class_cat[c] for c in all_classes],
               width, label="Categorical", color="steelblue", edgecolor="black")
bars2 = ax.bar(x + width/2, [per_class_ord[c] for c in all_classes],
               width, label="Ordinal", color="darkgreen", edgecolor="black")

ax.set_xlabel("Class")
ax.set_ylabel("Accuracy")
ax.set_title("Per-Class Accuracy: Categorical vs. Ordinal BART Model")
ax.set_xticks(x)
ax.set_xticklabels(all_classes, rotation=45, ha="right")
ax.set_ylim(0, 1.1)
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Overall accuracy bar chart
fig, ax = plt.subplots(figsize=(6, 4))
models = ["Categorical", "Ordinal"]
accs = [acc_cat, acc_ord]
bars = ax.bar(models, accs, color=["steelblue", "darkgreen"], edgecolor="black")
ax.set_ylabel("Accuracy")
ax.set_title("Overall Test Set Accuracy Comparison")
ax.set_ylim(0, 1)
for bar, a in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{a:.3f}", ha="center", va="bottom", fontweight="bold")
plt.tight_layout()
plt.show()

---

## Summary and Key Takeaways

This notebook demonstrated the `BARTModelWrapper` class for two types of classification:

### Model Types
- **Categorical (`target_type="categorical"`):** Treats target classes as unordered. Uses a BART prior with `dims=["classes", "n_obs"]` to produce a latent score matrix, passed through softmax into a `pm.Categorical` likelihood.
- **Ordinal (`target_type="ordinal"`):** Respects the natural ordering of classes. Uses a 1-D BART latent score with learned cutpoints and `pm.OrderedLogistic`.

### Key Concepts

1. **One-hot encoding:** PyMC-BART requires all-numeric inputs. The wrapper automatically one-hot encodes columns in `non_numeric_vars` via scikit-learn's `OneHotEncoder`.

2. **`register_data(df)`:** Pre-fits encoders on the full dataset before the train/test split, ensuring every category is known and consistently encoded.

3. **`X_shared_` (pm.Data container):** A PyTensor shared variable that holds the covariate matrix in the model graph. Its values can be swapped in-place with `set_value()` for out-of-sample prediction without rebuilding the model.

4. **"BART + likelihood jointly":** `pm.sample_posterior_predictive` draws from the **complete generative model** for each posterior sample — BART trees evaluate on new X to produce latent `mu`, the link function converts to class probabilities, and the likelihood samples predicted labels. This propagates full posterior uncertainty through the entire pipeline, not just the BART component.

### References
- [PyMC-BART documentation](https://www.pymc.io/projects/bart/en/latest/)
- [PyMC-BART categorical regression example](https://www.pymc.io/projects/bart/en/latest/examples/bart_categorical_hawks.html)
- [PyMC documentation on `pm.Data` and out-of-sample predictions](https://www.pymc.io/projects/docs/en/latest/learn/core_notebooks/pymc_overview.html)